# 인용 reference arXiv 메타데이터 수집: arXiv Atom API

`data/citations_ai`의 원본 논문에서 reference의 arXiv ID를 모읍니다. 같은 reference가 여러 원본 논문에 인용되면 reference당 한 행만 만들고, 원본 논문 ID는 `cit_arxiv_id` 리스트에 저장합니다.

arXiv Atom API의 `id_list`를 소규모 배치로 호출합니다. 각 배치가 끝날 때 캐시와 상태를 저장하므로 중단되어도 다음 실행에서 아직 조회하지 않은 ID만 계속 수집합니다. 최종 결과는 5,000행 단위 JSONL 파일입니다.

In [1]:
from pathlib import Path

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists() and (p / 'data' / 'citations_ai').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('프로젝트 루트 또는 notebooks/ 폴더에서 실행하세요.')

INPUT_FILES = [
    ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part1.jsonl',
    ROOT / 'data' / 'citations_ai' / 'arxiv_citations_part2.jsonl',
]
if missing := [path for path in INPUT_FILES if not path.exists()]:
    raise FileNotFoundError(missing)

OUTPUT_DIR = ROOT / 'data' / 'ai_references' / 'arxiv_ai_references_api'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FILE_PREFIX = 'arxiv_ai_references_api'
CACHE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_metadata_cache.jsonl'
STATE_PATH = OUTPUT_DIR / f'{FILE_PREFIX}_state.json'
CHUNK_SIZE = 5_000
API_BATCH_SIZE = 10  # 일시적인 rate limit·네트워크 실패를 줄이는 작은 id_list 배치
REQUEST_INTERVAL_SECONDS = 5.0
MAX_RETRIES = 5
MAX_RATE_LIMIT_RETRIES = 8
RATE_LIMIT_INITIAL_DELAY_SECONDS = 60.0
RUN_FULL_COLLECTION = True

print(f'입력: {len(INPUT_FILES)}개 / 출력: {OUTPUT_DIR}')


입력: 2개 / 출력: c:\Users\Playdata\Desktop\arxiv_graph_RAG\data\ai_references\arxiv_ai_references_api


In [2]:
import hashlib
import http.client
import json
import os
import re
import ssl
import tempfile
import time
import urllib.error
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
from datetime import datetime, timezone
from pathlib import Path
try:
    import truststore
except ImportError:
    truststore = None

ARXIV_API_URL = 'https://export.arxiv.org/api/query'
ARXIV_ID_RE = re.compile(r'^(?:arXiv:)?(?P<id>(?:\d{4}\.\d{4,5}|[A-Za-z-]+(?:\.[A-Za-z-]+)?/\d{7}))(?:v\d+)?$')
ATOM_NS = {'atom': 'http://www.w3.org/2005/Atom', 'arxiv': 'http://arxiv.org/schemas/atom'}
OUTPUT_FIELDS = ('id', 'title', 'abstract', 'authors', 'categories', 'primary_category', 'published', 'updated', 'doi', 'pdf_url', 'source')
_last_request_at = 0.0

def normalize_arxiv_id(value):
    if not isinstance(value, str):
        return None
    value = value.strip().removeprefix('https://arxiv.org/abs/').removeprefix('http://arxiv.org/abs/')
    match = ARXIV_ID_RE.fullmatch(value)
    return match.group('id') if match else None

def batched(items, size):
    for start in range(0, len(items), size):
        yield items[start:start + size]

def collect_references(files):
    collected, by_id, skipped = [], {}, 0
    for path in files:
        with Path(path).open(encoding='utf-8-sig') as handle:
            for line_number, line in enumerate(handle, 1):
                if not line.strip():
                    continue
                try:
                    row = json.loads(line)
                except json.JSONDecodeError as error:
                    raise ValueError(f'{path.name}:{line_number} JSON 파싱 실패') from error
                cit_id = normalize_arxiv_id(row.get('arxiv_id') if isinstance(row, dict) else None)
                if not cit_id:
                    raise ValueError(f'{path.name}:{line_number} 원본 arxiv_id 오류')
                references = row.get('references') or []
                if not isinstance(references, list):
                    raise ValueError(f'{path.name}:{line_number} references는 목록이어야 합니다.')
                for reference in references:
                    reference_id = normalize_arxiv_id(reference.get('arxiv_id') if isinstance(reference, dict) else None)
                    if not reference_id:
                        skipped += 1
                        continue
                    record = by_id.get(reference_id)
                    if record is None:
                        record = {'arxiv_id': reference_id, 'cit_arxiv_id': []}
                        by_id[reference_id] = record
                        collected.append(record)
                    if cit_id not in record['cit_arxiv_id']:
                        record['cit_arxiv_id'].append(cit_id)
    return collected, skipped

def parse_atom_entry(entry):
    arxiv_id = normalize_arxiv_id((entry.findtext('atom:id', namespaces=ATOM_NS) or '').rsplit('/abs/', 1)[-1])
    if not arxiv_id:
        return None
    categories = [node.attrib['term'] for node in entry.findall('atom:category', ATOM_NS) if node.attrib.get('term')]
    primary_category = entry.find('arxiv:primary_category', ATOM_NS)
    pdf_url = next((node.attrib.get('href') for node in entry.findall('atom:link', ATOM_NS) if node.attrib.get('title') == 'pdf'), f'https://arxiv.org/pdf/{arxiv_id}')
    return arxiv_id, {
        'id': f'https://arxiv.org/abs/{arxiv_id}',
        'title': ' '.join((entry.findtext('atom:title', namespaces=ATOM_NS) or '').split()),
        'abstract': ' '.join((entry.findtext('atom:summary', namespaces=ATOM_NS) or '').split()),
        'authors': [name for author in entry.findall('atom:author', ATOM_NS) if (name := author.findtext('atom:name', namespaces=ATOM_NS))],
        'categories': categories,
        'primary_category': primary_category.attrib.get('term') if primary_category is not None else (categories[0] if categories else None),
        'published': entry.findtext('atom:published', namespaces=ATOM_NS),
        'updated': entry.findtext('atom:updated', namespaces=ATOM_NS),
        'doi': entry.findtext('arxiv:doi', namespaces=ATOM_NS),
        'pdf_url': pdf_url,
        'source': 'arxiv',
    }

def make_output_record(reference, metadata, found):
    record = {'arxiv_id': reference['arxiv_id'], 'cit_arxiv_id': reference['cit_arxiv_id'], 'found': found}
    defaults = {'id': None, 'title': None, 'abstract': None, 'authors': [], 'categories': [], 'primary_category': None, 'published': None, 'updated': None, 'doi': None, 'pdf_url': None, 'source': 'arxiv'}
    record.update({field: (metadata or {}).get(field, defaults[field]) for field in OUTPUT_FIELDS})
    return record

def write_atomic(path, text):
    path = Path(path)
    with tempfile.NamedTemporaryFile(mode='w', encoding='utf-8', newline='\n', dir=path.parent, delete=False) as handle:
        temporary_path = Path(handle.name)
        handle.write(text)
    os.replace(temporary_path, path)

def append_jsonl(path, rows):
    if not rows:
        return
    with Path(path).open('a', encoding='utf-8', newline='\n') as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + '\n')

def load_cache(path):
    if not Path(path).exists():
        return {}
    with Path(path).open(encoding='utf-8') as handle:
        return {row['arxiv_id']: row for row in map(json.loads, filter(str.strip, handle))}

def input_signature(references):
    return hashlib.sha256('\n'.join(reference['arxiv_id'] for reference in references).encode()).hexdigest()

def save_state(path, **fields):
    write_atomic(path, json.dumps({**fields, 'updated_at': datetime.now(timezone.utc).isoformat()}, ensure_ascii=False, indent=2))

def describe_api_error(error):
    if isinstance(error, urllib.error.HTTPError):
        retry_after = error.headers.get('Retry-After') if error.headers else None
        try:
            body = error.read(300).decode('utf-8', errors='replace').replace('\n', ' ')
        except Exception:
            body = ''
        suffix = f'; Retry-After={retry_after}' if retry_after else ''
        return f'HTTP {error.code} {error.reason}{suffix}; body={body!r}'
    return f'{type(error).__name__}: {error}'

def retry_delay(error, attempt):
    if isinstance(error, urllib.error.HTTPError) and error.headers:
        value = error.headers.get('Retry-After')
        try:
            return min(max(float(value), REQUEST_INTERVAL_SECONDS), 600)
        except (TypeError, ValueError):
            pass
    return min(max(10 * (2 ** attempt), REQUEST_INTERVAL_SECONDS), 600)

def fetch_metadata_batch(arxiv_ids):
    global _last_request_at
    query = urllib.parse.urlencode({'id_list': ','.join(arxiv_ids), 'max_results': len(arxiv_ids)})
    request = urllib.request.Request(f'{ARXIV_API_URL}?{query}', headers={'User-Agent': 'arxiv-reference-harvester/1.0 (contact: local-notebook)'})
    ordinary_attempt = 0
    rate_limit_attempt = 0
    while ordinary_attempt <= MAX_RETRIES:
        wait = REQUEST_INTERVAL_SECONDS - (time.monotonic() - _last_request_at)
        if wait > 0:
            time.sleep(wait)
        try:
            _last_request_at = time.monotonic()
            context = truststore.SSLContext(ssl.PROTOCOL_TLS_CLIENT) if truststore else ssl.create_default_context()
            with urllib.request.urlopen(request, timeout=90, context=context) as response:
                root = ET.fromstring(response.read())
            return dict(filter(None, (parse_atom_entry(entry) for entry in root.findall('atom:entry', ATOM_NS))))
        except (urllib.error.HTTPError, urllib.error.URLError, TimeoutError, ConnectionError, http.client.HTTPException, ET.ParseError) as error:
            detail = describe_api_error(error)
            if isinstance(error, urllib.error.HTTPError) and error.code == 429:
                rate_limit_attempt += 1
                if rate_limit_attempt > MAX_RATE_LIMIT_RETRIES:
                    raise RuntimeError(f'arXiv API rate limit 지속: {arxiv_ids[:3]}... / 마지막 원인: {detail}') from error
                delay = retry_delay(error, rate_limit_attempt - 1)
                if not (error.headers and error.headers.get('Retry-After')):
                    delay = min(max(RATE_LIMIT_INITIAL_DELAY_SECONDS * (2 ** (rate_limit_attempt - 1)), delay), 900)
            else:
                if ordinary_attempt >= MAX_RETRIES:
                    raise RuntimeError(f'arXiv API 요청 실패: {arxiv_ids[:3]}... / 마지막 원인: {detail}') from error
                delay = retry_delay(error, ordinary_attempt)
                ordinary_attempt += 1
            print(f'요청 실패 {detail}: {delay:.0f}초 후 재시도')
            time.sleep(delay)


In [3]:
references, skipped = collect_references(INPUT_FILES)
signature = input_signature(references)
state = json.loads(STATE_PATH.read_text(encoding='utf-8')) if STATE_PATH.exists() else {}
if state and state.get('input_signature') != signature:
    raise RuntimeError('입력 파일이 이전 실행과 달라졌습니다. 기존 출력 폴더를 백업 또는 비운 뒤 다시 실행하세요.')

cache = load_cache(CACHE_PATH)
pending_ids = [reference['arxiv_id'] for reference in references if reference['arxiv_id'] not in cache]
print(f'고유 reference: {len(references):,}; arXiv ID 없음: {skipped:,}; 이미 캐시됨: {len(cache):,}; 남은 조회: {len(pending_ids):,}')

if RUN_FULL_COLLECTION:
    previous_batches = state.get('batches_completed', state.get('last_completed_batch', 0))
    for batch_index, batch_ids in enumerate(batched(pending_ids, API_BATCH_SIZE), 1):
        metadata_by_id = fetch_metadata_batch(batch_ids)
        new_cache_rows = [
            {'arxiv_id': arxiv_id, 'found': arxiv_id in metadata_by_id, 'metadata': metadata_by_id.get(arxiv_id)}
            for arxiv_id in batch_ids
        ]
        append_jsonl(CACHE_PATH, new_cache_rows)
        cache.update({row['arxiv_id']: row for row in new_cache_rows})
        save_state(STATE_PATH, input_signature=signature, harvest_complete=False, reference_count=len(references), cached_count=len(cache), remaining_count=len(references) - len(cache), last_completed_batch=previous_batches + batch_index, batches_completed=previous_batches + batch_index)
        if batch_index % 10 == 0 or batch_index * API_BATCH_SIZE >= len(pending_ids):
            print(f'{batch_index:,}개 새 배치 완료 / 캐시 {len(cache):,}/{len(references):,}')

if len(cache) != len(references):
    raise RuntimeError('수집이 완료되지 않았습니다. 상태는 저장되었으며 같은 셀을 다시 실행하면 이어서 수집합니다.')

records = [make_output_record(reference, cache[reference['arxiv_id']]['metadata'], cache[reference['arxiv_id']]['found']) for reference in references]
for index, rows in enumerate(batched(records, CHUNK_SIZE), 1):
    write_atomic(OUTPUT_DIR / f'{FILE_PREFIX}_part{index}.jsonl', ''.join(json.dumps(row, ensure_ascii=False) + '\n' for row in rows))
save_state(STATE_PATH, input_signature=signature, harvest_complete=True, reference_count=len(references), cached_count=len(cache), remaining_count=0, output_rows=len(records))
print(f'최종 저장 완료: {len(records):,}행, {((len(records) - 1) // CHUNK_SIZE) + 1:,}개 파일')


고유 reference: 31,913; arXiv ID 없음: 60,549; 이미 캐시됨: 10,885; 남은 조회: 21,028
10개 새 배치 완료 / 캐시 10,985/31,913
20개 새 배치 완료 / 캐시 11,085/31,913
30개 새 배치 완료 / 캐시 11,185/31,913
40개 새 배치 완료 / 캐시 11,285/31,913
50개 새 배치 완료 / 캐시 11,385/31,913
60개 새 배치 완료 / 캐시 11,485/31,913
70개 새 배치 완료 / 캐시 11,585/31,913
80개 새 배치 완료 / 캐시 11,685/31,913
90개 새 배치 완료 / 캐시 11,785/31,913
100개 새 배치 완료 / 캐시 11,885/31,913
110개 새 배치 완료 / 캐시 11,985/31,913
120개 새 배치 완료 / 캐시 12,085/31,913
130개 새 배치 완료 / 캐시 12,185/31,913
140개 새 배치 완료 / 캐시 12,285/31,913
150개 새 배치 완료 / 캐시 12,385/31,913
160개 새 배치 완료 / 캐시 12,485/31,913
170개 새 배치 완료 / 캐시 12,585/31,913
180개 새 배치 완료 / 캐시 12,685/31,913
190개 새 배치 완료 / 캐시 12,785/31,913
200개 새 배치 완료 / 캐시 12,885/31,913
210개 새 배치 완료 / 캐시 12,985/31,913
220개 새 배치 완료 / 캐시 13,085/31,913
230개 새 배치 완료 / 캐시 13,185/31,913
240개 새 배치 완료 / 캐시 13,285/31,913
250개 새 배치 완료 / 캐시 13,385/31,913
260개 새 배치 완료 / 캐시 13,485/31,913
270개 새 배치 완료 / 캐시 13,585/31,913
280개 새 배치 완료 / 캐시 13,685/31,913
290개 새 배치 완료 / 캐시 13,785/31,913
요청 실패 HT

In [4]:
if STATE_PATH.exists() and json.loads(STATE_PATH.read_text(encoding='utf-8')).get('harvest_complete'):
    saved = []
    for path in sorted(OUTPUT_DIR.glob(f'{FILE_PREFIX}_part*.jsonl')):
        with path.open(encoding='utf-8') as handle:
            saved.extend(json.loads(line) for line in handle if line.strip())
    assert len(saved) == len(references) == len({row['arxiv_id'] for row in saved})
    assert {row['arxiv_id']: row['cit_arxiv_id'] for row in saved} == {row['arxiv_id']: row['cit_arxiv_id'] for row in references}
    assert all(isinstance(row['cit_arxiv_id'], list) and row['cit_arxiv_id'] for row in saved)
    assert all(row['title'] and row['abstract'] for row in saved if row['found'])
    print(f'검증 완료: {len(saved):,}개 reference / {sum(len(row["cit_arxiv_id"]) for row in saved):,}개 인용 연결')
else:
    print('수집이 아직 끝나지 않았습니다. 완료 후 이 셀을 실행하면 최종 결과를 검증합니다.')


검증 완료: 31,913개 reference / 69,438개 인용 연결


## API 연결 소량 테스트

전체 수집과 별개로 Atom API 응답과 파서를 확인합니다. 성공하면 `found=True` 및 제목이 출력됩니다.

In [5]:
test_ids = ['2301.00001', '2301.00002', '2301.00003']
test_metadata = fetch_metadata_batch(test_ids)
assert test_metadata, 'arXiv API에서 논문 메타데이터를 받지 못했습니다.'
assert all(metadata['title'] and metadata['abstract'] for metadata in test_metadata.values())
print(f'API 연결 성공: 요청 {len(test_ids)}개 중 응답 {len(test_metadata)}개')
for arxiv_id, metadata in test_metadata.items():
    print(arxiv_id, '-', metadata['title'])


API 연결 성공: 요청 3개 중 응답 3개
2301.00001 - NFTrig
2301.00003 - Emotion in Cognitive Architecture: Emergent Properties from Interactions with Human Emotion
2301.00002 - Evaluating Alternative Glyph Design for Showing Large-Magnitude-Range Quantum Spins
